# 05. Condições Meteorológicas por Foco

Para cada foco INPE, extrai as condições meteorológicas ERA5 no pixel de grade mais próximo e no dia da detecção.

**Entrada:** `data/inpe_queimadas/inpe_focos_2023_consolidado.csv` + `data/era5/era5_2023_MM_brasil.nc`
**Saída:** `data/era5/focos_era5_2023.csv` — mesmas 189.901 linhas com colunas ERA5 adicionadas
**Método:** seleção vetorizada por dia (`xr.Dataset.sel(method='nearest')`) — sem loop por foco

## 0. Dependências e configuração

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

ERA5_DIR  = Path('data/era5')
FOCOS_DIR = Path('data/inpe_queimadas')
ANO       = 2023
MESES_PT  = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']

print('Configuracao OK')
print(f'ERA5_DIR : {ERA5_DIR}')
print(f'Arquivos ERA5: {len(list(ERA5_DIR.glob(f"era5_{ANO}_*_brasil.nc")))} / 12')

## 1. Carregamento dos focos

In [ ]:
df_focos = pd.read_csv(
    FOCOS_DIR / f'inpe_focos_{ANO}_consolidado.csv',
    usecols=['latitude', 'longitude', 'data_hora_gmt', 'data_pas', 'bioma', 'frp', 'mes'],
    parse_dates=['data_hora_gmt', 'data_pas'],
    low_memory=False,
)
df_focos['data_pas'] = pd.to_datetime(df_focos['data_pas']).dt.normalize()

print(f'Focos: {len(df_focos):,}')
print(f'Periodo: {df_focos["data_pas"].min().date()} -> {df_focos["data_pas"].max().date()}')
print(f'Meses: {sorted(df_focos["mes"].unique())}')
df_focos.head(3)

## 2. Extração ERA5 por foco (mês a mês)

Para cada mês:
1. Carrega o NetCDF mensal ERA5 (4 snapshots/dia: 00, 06, 12, 18 UTC)
2. Calcula variáveis derivadas e média diária
3. Para cada dia com focos, seleciona todos os pixels mais próximos de uma vez (vetorizado)
4. Descarta o Dataset do mês antes de avançar (memória)

In [ ]:
out_cache = ERA5_DIR / f'focos_era5_{ANO}.csv'

if out_cache.exists():
    df_era5_focos = pd.read_csv(out_cache, parse_dates=['data_pas'])
    print(f'[cache] {out_cache.name}  ({len(df_era5_focos):,} focos)')
else:
    partes   = []
    meses_ok = 0

    for mes in range(1, 13):
        arq = ERA5_DIR / f'era5_{ANO}_{mes:02d}_brasil.nc'
        if not arq.exists():
            print(f'  [faltando] {arq.name} — mes {MESES_PT[mes-1]} sera ignorado')
            continue

        focos_mes = df_focos[df_focos['mes'] == mes].copy()
        if len(focos_mes) == 0:
            continue

        ds = xr.open_dataset(arq)

        # Variaveis derivadas
        ws  = np.sqrt(ds['u10']**2 + ds['v10']**2)
        tc  = ds['t2m'] - 273.15
        td  = ds['d2m'] - 273.15
        rh  = 100 * (np.exp((17.625 * td) / (243.04 + td)) /
                     np.exp((17.625 * tc) / (243.04 + tc)))
        eta = 1 - 2 * (rh / 100) + (rh / 100) ** 2
        ff  = ws * np.sqrt(eta)

        ds_d = xr.Dataset({
            'wind_speed': ws.resample(time='1D').mean(),
            'temp_c'    : tc.resample(time='1D').mean(),
            'rh'        : rh.resample(time='1D').mean(),
            'ffwi'      : ff.resample(time='1D').mean(),
        })

        # Extracao vetorizada: um sel por dia (todos os focos do dia de uma vez)
        partes_mes = []
        for date_val in sorted(focos_mes['data_pas'].unique()):
            focos_dia = focos_mes[focos_mes['data_pas'] == date_val].copy()
            lats = xr.DataArray(focos_dia['latitude'].values,  dims='points')
            lons = xr.DataArray(focos_dia['longitude'].values, dims='points')
            try:
                era5_dia = ds_d.sel(time=date_val, method='nearest')
                pts = era5_dia.sel(latitude=lats, longitude=lons, method='nearest')
                focos_dia['wind_speed_era5'] = pts['wind_speed'].values
                focos_dia['temp_c_era5']     = pts['temp_c'].values
                focos_dia['rh_era5']         = pts['rh'].values
                focos_dia['ffwi_era5']       = pts['ffwi'].values
            except Exception:
                for col in ['wind_speed_era5', 'temp_c_era5', 'rh_era5', 'ffwi_era5']:
                    focos_dia[col] = np.nan
            partes_mes.append(focos_dia)

        partes.append(pd.concat(partes_mes, ignore_index=True))
        ds.close()
        meses_ok += 1
        print(f'  {MESES_PT[mes-1]:>3}: {len(focos_mes):,} focos  OK')

    df_era5_focos = pd.concat(partes, ignore_index=True)
    df_era5_focos.to_csv(out_cache, index=False)
    print(f'\n[ok] {out_cache.name}  ({len(df_era5_focos):,} focos | {meses_ok}/12 meses)')

## 3. Verificação

In [ ]:
cols_era5 = ['wind_speed_era5', 'temp_c_era5', 'rh_era5', 'ffwi_era5']

print(f'Shape: {df_era5_focos.shape}')
print('\nPreenchimento:')
for col in cols_era5:
    pct = df_era5_focos[col].notna().mean() * 100
    print(f'  {col:<22}: {pct:.1f}%')

print()
print(df_era5_focos[cols_era5].describe().round(2).to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle(f'ERA5 por Foco — Distribuicoes {ANO}', fontsize=13, fontweight='bold')

meta = [
    ('temp_c_era5',     'Temperatura no Foco (deg C)',      '#d62728'),
    ('rh_era5',         'Umidade Relativa no Foco (%)',     '#2ca02c'),
    ('wind_speed_era5', 'Velocidade do Vento no Foco (m/s)','#1f77b4'),
    ('ffwi_era5',       'FFWI no Foco',                    '#ff7f0e'),
]
for ax, (col, titulo, cor) in zip(axes.flat, meta):
    data = df_era5_focos[col].dropna()
    ax.hist(data, bins=60, color=cor, alpha=0.8, edgecolor='white')
    ax.axvline(data.median(), color='black', linestyle='--', linewidth=1,
               label=f'Mediana: {data.median():.1f}')
    ax.set_title(titulo, fontsize=10)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.25)

plt.tight_layout()
plt.savefig(ERA5_DIR / f'fig_era5_por_foco_{ANO}.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Exportação

In [ ]:
print(f'Arquivo salvo: {out_cache}')
print(f'Tamanho: {out_cache.stat().st_size / 1e6:.1f} MB')
print(f'Colunas ERA5 adicionadas: {cols_era5}')
print('Pronto para os notebooks 06 e 07 (velocidade e consolidacao).')